# NFIP Single-Residence Severity — Data Wrangling

Builds `model_ready_nfip.parquet` from three sources:

- **OpenFEMA NFIP Redacted Claims v2** — claim-level records
- **NHGIS / IPUMS** — ACS median home value at census block-group level
- **NOAA coastal county classification** — shoreline and watershed designations

Population: single-family residential and mobile-home claims on policies first
written 2011 or later.

## Design notes

1. **Sentinel codes are recoded before any missing-value fill.** NFIP elevation
   fields carry fill codes (`9999`, `9990.0`, `±99999.9`) as ordinary numbers, so
   filling nulls alone would leave them in the design matrix as observed
   9,990-foot elevations. Order matters: recode, then flag, then fill.
2. **Census jam values and top-codes are handled separately.** `-666666666`
   ("estimate could not be computed") becomes `NaN`; `1000001` ("$1,000,000 or
   more") is a genuine censored observation, retained with an indicator column.
   Dropping top-codes would systematically delete high-value coastal tracts.
3. **`lowestFloorElevation` is retained.** Per the OpenFEMA data dictionary it is
   the rating elevation and is not redundant with the other elevation fields.
4. **A validation suite runs before export** and raises on failure, so a defect
   surfaces here rather than as a convergence error downstream.
5. **Non-positive `buildingDamageAmount` is flagged, not silently carried.**
   Gamma and lognormal severity models require a strictly positive response.

## Data placement

Paths are set in the configuration cell below. Expected layout under `BASE`:

```
FimaNfipClaimsV2.parquet          OpenFEMA claims file
county_classification.csv         NOAA coastal counties (in this repo)
nhgis/*_blck_grp.csv              NHGIS ACS block-group extracts
Single-Residence Severity/
    data/processed/               output written here
```

Change `BASE` to match your environment. The Colab mount can be removed if
running locally.

## Coastal classification

`county_classification.csv` is built from NOAA's *Defining Coastal Counties*
reference (2010 boundaries). The definitions are **nested**: all 769 counties in
the file are coastal watershed counties, and 452 of those are also coastal
shoreline counties. Counties absent from the file are non-coastal. Section 2
spot-checks a list of unambiguously coastal counties and sets
`SPATIAL_FLAGS_TRUSTED` accordingly.

## 1. Configuration

In [ ]:
import os
import re
import glob
import datetime
import warnings

import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

BASE          = '/content/drive/MyDrive/Programming/nfip'
CLAIMS_PATH   = f'{BASE}/FimaNfipClaimsV2.parquet'
COUNTY_PATH   = f'{BASE}/county_classification.csv'
NHGIS_DIR     = f'{BASE}/nhgis'
REPO_DIR      = f'{BASE}/Single-Residence Severity'
PROCESSED_DIR = f'{REPO_DIR}/data/processed'
OUT_PATH      = f'{PROCESSED_DIR}/model_ready_nfip.parquet'

START_DATE = datetime.date(2011, 1, 1)

# Sentinel / null codes.
#
# elevationDifference 9999 is DOCUMENTED in the OpenFEMA FimaNfipClaims v2 data
# dictionary: "The value of 9999.0 indicates the field is not reported and/or used
# for this policy."
#
# The remaining codes are EMPIRICALLY INFERRED, not documented. baseFloodElevation
# and lowestAdjacentGrade are typed decimal(6,1), whose maximum magnitude is exactly
# 99999.9 -- the observed extreme sits precisely at the type boundary, which is a
# system fill rather than a measurement. Values are recoded by magnitude threshold
# rather than exact match because the codes cluster (9990.0, 9980.0, 9999.9 all
# appear), so an exact-value list keeps finding one more.
ELEV_SENTINEL_MIN = 9000.0     # |value| >= this is treated as a fill code
ELEV_DIFF_DOCUMENTED_NULL = 9999

# Census ACS codes
ACS_JAM_VALUE  = -666666666.0  # "estimate could not be computed" -> missing
ACS_TOP_CODE   = 1000001.0     # "$1,000,000 or more" -> real, but censored above

pd.set_option('display.max_columns', 60)
print('Config loaded.')

Mounted at /content/drive
Config loaded.


## 2. County classification — load and validate

The merge logic here is sound: `FIPS` is read as string so leading zeros survive,
and the claims-side key is zero-padded to 5 characters. What follows is a content
check on the lookup table itself, using counties whose coastal status is not in
dispute.

In [ ]:
county_classes_df = pd.read_csv(COUNTY_PATH, dtype={'FIPS': str})
print(f'Loaded {len(county_classes_df):,} county records.')

# Counties that are unambiguously on the open coast. If any of these read 0 for
# SHORELINE_FLAG, the lookup table is miscoded.
COASTAL_SPOT_CHECK = {
    '01003': 'Baldwin County, AL',
    '01097': 'Mobile County, AL',
    '34029': 'Ocean County, NJ',
    '34009': 'Cape May County, NJ',
    '34025': 'Monmouth County, NJ',
    '45019': 'Charleston County, SC',
    '09001': 'Fairfield County, CT',
    '06037': 'Los Angeles County, CA',
    '12086': 'Miami-Dade County, FL',
    '48201': 'Harris County, TX',
}

lookup = county_classes_df.set_index('FIPS')['SHORELINE_FLAG'].to_dict()
failures = []
for fips, name in COASTAL_SPOT_CHECK.items():
    val = lookup.get(fips, 'MISSING FROM TABLE')
    status = 'OK' if val == 1 else 'FAIL'
    if status == 'FAIL':
        failures.append((fips, name, val))
    print(f'  [{status}] {name:<28} FIPS {fips}  SHORELINE_FLAG = {val}')

if failures:
    warnings.warn(
        f'\n{len(failures)} of {len(COASTAL_SPOT_CHECK)} coastal spot-checks failed. '
        'county_classification.csv is miscoded and must be rebuilt from the NOAA '
        'coastal shoreline county list. SHORELINE_FLAG / WATERSHED_FLAG should be '
        'excluded from modelling until then.',
        UserWarning
    )
    SPATIAL_FLAGS_TRUSTED = False
else:
    SPATIAL_FLAGS_TRUSTED = True

print(f'\nSPATIAL_FLAGS_TRUSTED = {SPATIAL_FLAGS_TRUSTED}')
print('\nFlag coverage by state (top 15 by county count):')
print(county_classes_df.groupby('STATE')['SHORELINE_FLAG']
      .agg(['sum', 'count']).sort_values('count', ascending=False).head(15))

Loaded 3,235 county records.
  [FAIL] Baldwin County, AL           FIPS 01003  SHORELINE_FLAG = 0
  [FAIL] Mobile County, AL            FIPS 01097  SHORELINE_FLAG = 0
  [FAIL] Ocean County, NJ             FIPS 34029  SHORELINE_FLAG = 0
  [FAIL] Cape May County, NJ          FIPS 34009  SHORELINE_FLAG = 0
  [FAIL] Monmouth County, NJ          FIPS 34025  SHORELINE_FLAG = 0
  [FAIL] Charleston County, SC        FIPS 45019  SHORELINE_FLAG = 0
  [FAIL] Fairfield County, CT         FIPS 09001  SHORELINE_FLAG = 0
  [FAIL] Los Angeles County, CA       FIPS 06037  SHORELINE_FLAG = 0
  [FAIL] Miami-Dade County, FL        FIPS 12086  SHORELINE_FLAG = 0
  [FAIL] Harris County, TX            FIPS 48201  SHORELINE_FLAG = 0

SPATIAL_FLAGS_TRUSTED = False

Flag coverage by state (top 15 by county count):
       sum  count
STATE            
TX      11    254
GA       8    159
VA      32    134
KY       0    120
MO       0    115
KS       0    105
IL       2    102
NC       4    100
IA       0     99
TN

/tmp/ipykernel_1012/3219583828.py:29: UserWarning: 
10 of 10 coastal spot-checks failed. county_classification.csv is miscoded and must be rebuilt from the NOAA coastal shoreline county list. SHORELINE_FLAG / WATERSHED_FLAG should be excluded from modelling until then.
  warnings.warn(


## 3. Load claims and apply population filters

In [ ]:
COLUMNS_TO_LOAD = [
    'primaryResidenceIndicator', 'rentalPropertyIndicator',
    'basementEnclosureCrawlspaceType', 'crsClassificationCode',
    'elevatedBuildingIndicator', 'elevationDifference', 'baseFloodElevation',
    'ratedFloodZone', 'lowestAdjacentGrade', 'lowestFloorElevation',
    'numberOfFloorsInTheInsuredBuilding', 'obstructionType', 'occupancyType',
    'originalConstructionDate', 'originalNBDate', 'postFIRMConstructionIndicator',
    'floodproofedIndicator', 'floodZoneCurrent', 'buildingDescriptionCode',
    'countyCode', 'censusTract', 'censusBlockGroupFips',
    'amountPaidOnBuildingClaim', 'buildingDamageAmount',
    'netBuildingPaymentAmount', 'buildingPropertyValue',
    'buildingReplacementCost', 'asOfDate', 'dateOfLoss',
]

nfip = pd.read_parquet(
    CLAIMS_PATH,
    columns=COLUMNS_TO_LOAD,
    filters=[('originalNBDate', '>=', START_DATE)],
)
print(f'Loaded {len(nfip):,} claims with originalNBDate >= {START_DATE}.')

# Single-family residential + mobile homes
nfip = nfip[nfip['occupancyType'].isin([1, 11, 14, '1', '11', '14'])]
nfip = nfip[nfip['buildingDescriptionCode'].isin([1, 18, 19, '01', '1', '18', '19'])]
print(f'After occupancy / building-description filters: {len(nfip):,}')

nfip['originalNBDate'] = pd.to_datetime(nfip['originalNBDate'], errors='coerce')
nfip['dateOfLoss']     = pd.to_datetime(nfip['dateOfLoss'], errors='coerce')
nfip['uw_year']        = nfip['originalNBDate'].dt.year
nfip['yearOfLoss']     = nfip['dateOfLoss'].dt.year

nfip = nfip.dropna(subset=['buildingDamageAmount'])
print(f'After dropping null buildingDamageAmount: {len(nfip):,}')

n_nonpos = (nfip['buildingDamageAmount'] <= 0).sum()
print(f'\nNon-positive buildingDamageAmount: {n_nonpos:,} '
      f'({n_nonpos / len(nfip) * 100:.2f}%)')
print('These are RETAINED here and flagged. Gamma / lognormal severity models must '
      'exclude them at fit time; Tweedie with 1 < p < 2 can use them.')
nfip['is_zero_damage'] = (nfip['buildingDamageAmount'] <= 0).astype(int)

Loaded 490,076 claims with originalNBDate >= 2011-01-01.
After occupancy / building-description filters: 341,527
After dropping null buildingDamageAmount: 292,109

Non-positive buildingDamageAmount: 11,414 (3.91%)
These are RETAINED here and flagged. Gamma / lognormal severity models must exclude them at fit time; Tweedie with 1 < p < 2 can use them.


## 4. Sentinel recoding — the critical fix

This runs **before** any `fillna`. Order matters: a sentinel code is an ordinary
number, not a null, so filling nulls first leaves `9990.0` in the design matrix as
an observed 9,990-foot elevation.

In [ ]:
ELEV_COLS = ['elevationDifference', 'baseFloodElevation',
             'lowestAdjacentGrade', 'lowestFloorElevation']

print('--- Before recoding ---')
for c in ELEV_COLS:
    s = pd.to_numeric(nfip[c], errors='coerce')
    print(f'{c:<24} min {s.min():>12.1f}  max {s.max():>12.1f}  '
          f'|v|>={ELEV_SENTINEL_MIN:.0f}: {(s.abs() >= ELEV_SENTINEL_MIN).sum():,}')

for c in ELEV_COLS:
    nfip[c] = pd.to_numeric(nfip[c], errors='coerce')

# Documented null code for elevationDifference
n_doc = (nfip['elevationDifference'] == ELEV_DIFF_DOCUMENTED_NULL).sum()
nfip.loc[nfip['elevationDifference'] == ELEV_DIFF_DOCUMENTED_NULL,
         'elevationDifference'] = np.nan
print(f'\nRecoded {n_doc:,} documented 9999 codes in elevationDifference.')

# Magnitude-threshold fill codes
for c in ELEV_COLS:
    mask = nfip[c].abs() >= ELEV_SENTINEL_MIN
    print(f'Recoded {mask.sum():,} fill codes in {c}.')
    nfip.loc[mask, c] = np.nan

print('\n--- After recoding ---')
for c in ELEV_COLS:
    s = nfip[c]
    print(f'{c:<24} min {s.min():>10.1f}  max {s.max():>10.1f}  '
          f'null {s.isna().sum():>8,} ({s.isna().mean() * 100:5.1f}%)')

# Year-in-elevation-field data entry errors. These are NOT sentinels -- they are
# a corrupted BFE (a year written into an elevation field) propagating into the
# derived elevationDifference. Flagged rather than deleted, since a genuine
# high-elevation inland property can legitimately exceed 900 ft.
year_like = nfip['baseFloodElevation'].between(1900, 2030)
print(f'\nbaseFloodElevation values in year range 1900-2030: {year_like.sum():,}')
if year_like.sum():
    print(nfip.loc[year_like, 'baseFloodElevation'].value_counts().head(10))
    print('\nReview these manually. Values clustering on plausible years '
          '(1970-2005) are data-entry errors; scattered values may be real '
          'high-elevation inland properties.')

--- Before recoding ---
elevationDifference      min      -1978.0  max        987.0  |v|>=9000: 0
baseFloodElevation       min        -13.8  max       9990.0  |v|>=9000: 1,225
lowestAdjacentGrade      min     -99999.9  max       9998.7  |v|>=9000: 77
lowestFloorElevation     min        -25.0  max       9998.0  |v|>=9000: 609

Recoded 0 documented 9999 codes in elevationDifference.
Recoded 0 fill codes in elevationDifference.
Recoded 1,225 fill codes in baseFloodElevation.
Recoded 77 fill codes in lowestAdjacentGrade.
Recoded 609 fill codes in lowestFloorElevation.

--- After recoding ---
elevationDifference      min    -1978.0  max      987.0  null  188,314 ( 64.5%)
baseFloodElevation       min      -13.8  max     7942.1  null  190,384 ( 65.2%)
lowestAdjacentGrade      min      -17.0  max     7962.4  null  191,870 ( 65.7%)
lowestFloorElevation     min      -25.0  max     7962.8  null  191,122 ( 65.4%)

baseFloodElevation values in year range 1900-2030: 33
baseFloodElevation
1952.0    5

## 5. Feature engineering

In [ ]:
# This cell is idempotent -- safe to re-run without re-running section 3.

# Mobile homes. Guarded because the source columns are dropped below; without the
# guard, a re-run raises KeyError: 'occupancyType'.
if 'occupancyType' in nfip.columns:
    nfip['is_mobile_home'] = (
        nfip['occupancyType'].isin([14, '14', 14.0]) |
        nfip['buildingDescriptionCode'].isin([18, 19, '18', '19', 18.0, 19.0])
    ).astype(int)
    nfip.loc[nfip['is_mobile_home'] == 1, 'numberOfFloorsInTheInsuredBuilding'] = 1.0
    nfip = nfip.drop(columns=['occupancyType', 'buildingDescriptionCode'])
else:
    print('is_mobile_home already derived; skipping (cell re-run).')

# Flood zone. Zone D is folded into Unknown at source: it means "undetermined
# hazard," and it carries ~33 records post-filter, too few to estimate. The
# Note that RENAMING the level (e.g. to 'Other') does not help -- the sparse cell
# survives under a new name and still breaks the fit. It must be merged.
def map_flood_zone(zone):
    if pd.isna(zone):
        return 'Unknown'
    z = str(zone).upper().strip()
    if z.startswith('V'):
        return 'V Zones'
    if z.startswith('A'):
        return 'A Zones'
    if z.startswith('D'):
        return 'Unknown'
    if z == 'B':
        return 'Zone B & X (shaded)'
    if z in ('C', 'X'):
        return 'Zone C & X (unshaded)'
    return 'Unknown'

nfip['floodZoneCurrent'] = nfip['floodZoneCurrent'].apply(map_flood_zone)
nfip = nfip.drop(columns=[c for c in ('ratedFloodZone', 'crsClassificationCode')
                          if c in nfip.columns])

def map_obstruction(obs):
    if pd.isna(obs):
        return 'Unknown'
    val = str(obs).strip()
    if val in ('10', '10.0', '1'):
        return 'Free of Obstruction'
    if val == 'Unknown':
        return 'Unknown'
    return 'Obstructed'

nfip['obstructionType'] = nfip['obstructionType'].fillna('Unknown').apply(map_obstruction)

def map_basement(val):
    if pd.isna(val):
        return 'Unknown'
    v = str(val).strip()
    if v.endswith('.0'):
        v = v[:-2]
    return {'0': 'None', '1': 'Finished Basement', '2': 'Unfinished Basement',
            '3': 'Crawlspace', '4': 'Subgrade Crawlspace'}.get(v, 'Unknown')

nfip['basementEnclosureCrawlspaceType'] = (
    nfip['basementEnclosureCrawlspaceType'].apply(map_basement))

for col in ('primaryResidenceIndicator', 'rentalPropertyIndicator',
            'elevatedBuildingIndicator'):
    nfip[col] = nfip[col].map({
        True: 'True', False: 'False', 'True': 'True', 'False': 'False',
        1: 'True', 0: 'False', 1.0: 'True', 0.0: 'False',
    }).fillna('Unknown')

nfip['numberOfFloorsInTheInsuredBuilding'] = (
    nfip['numberOfFloorsInTheInsuredBuilding'].fillna(1.0))

# Building age at policy inception.
#
# Computed via decimal years rather than Timedelta subtraction. datetime64[ns]
# differences are stored as int64 nanoseconds, which overflows on this data --
# construction dates run back to the 1780s, and (2020 - 1780) in nanoseconds
# exceeds the int64 range. Float year arithmetic sidesteps the issue entirely.
nfip['originalConstructionDate'] = pd.to_datetime(
    nfip['originalConstructionDate'], errors='coerce')

def _decimal_year(s):
    s = pd.to_datetime(s, errors='coerce')
    return s.dt.year + (s.dt.dayofyear - 1) / 365.25

nfip['buildingAge'] = (_decimal_year(nfip['originalNBDate'])
                       - _decimal_year(nfip['originalConstructionDate']))

n_neg = (nfip['buildingAge'] < 0).sum()
n_old = (nfip['buildingAge'] > 250).sum()
nfip.loc[nfip['buildingAge'] < 0, 'buildingAge'] = np.nan
nfip.loc[nfip['buildingAge'] > 250, 'buildingAge'] = np.nan
print(f'\nbuildingAge: {n_neg:,} negative and {n_old:,} over 250 years set to NaN.')
print(f'  range: {nfip["buildingAge"].min():.1f} to {nfip["buildingAge"].max():.1f} years, '
      f'{nfip["buildingAge"].isna().sum():,} null')
print('  Note: ages of 200+ years are plausible for New England housing stock and '
      'are retained.')

print('Category distributions:')
for c in ('floodZoneCurrent', 'basementEnclosureCrawlspaceType', 'obstructionType'):
    print(f'\n{c}:')
    print(nfip[c].value_counts())


buildingAge: 1,335 negative and 6 over 250 years set to NaN.
  range: 0.0 to 249.9 years, 1,409 null
  Note: ages of 200+ years are plausible for New England housing stock and are retained.
Category distributions:

floodZoneCurrent:
floodZoneCurrent
A Zones                  171415
Zone C & X (unshaded)     69842
Unknown                   41540
V Zones                    5360
Zone B & X (shaded)        3952
Name: count, dtype: int64

basementEnclosureCrawlspaceType:
basementEnclosureCrawlspaceType
Unknown                141603
None                    90014
Unfinished Basement     27116
Finished Basement       23534
Subgrade Crawlspace      9842
Name: count, dtype: int64

obstructionType:
obstructionType
Unknown                209110
Obstructed              61213
Free of Obstruction     21786
Name: count, dtype: int64


## 6. Missingness indicators and elevation fill

Order matters: sentinels are already `NaN` from section 4, so the indicators below
capture genuine missingness rather than mislabelling a 9,990-foot fill code as an
observed elevation.

The three indicators are near-duplicates of each other (they agreed on 99.6% of
rows in the prior run), so only **one** should enter any single GLM. `is_elev_missing`
below is the consolidated version.

In [ ]:
# This cell is idempotent -- and the guard is load-bearing, not cosmetic. The
# fillna(0) at the bottom means a re-run would recompute every flag as all-zero
# with no error raised, silently destroying the missingness indicators.
_FLAGS_EXIST = 'is_elev_missing' in nfip.columns
if _FLAGS_EXIST:
    print('Missingness flags already computed; skipping recomputation (cell re-run).')
    print('To rebuild them, re-run sections 3-5 first.')

for c, flag in [('lowestAdjacentGrade', 'is_lag_missing'),
                ('elevationDifference', 'is_elev_diff_missing'),
                ('baseFloodElevation',  'is_bfe_missing'),
                ('lowestFloorElevation','is_lfe_missing')]:
    if not _FLAGS_EXIST:
        nfip[flag] = nfip[c].isna().astype(int)

# Consolidated indicator -- use this one in models; the individual flags are
# retained for diagnostics only.
if not _FLAGS_EXIST:
    nfip['is_elev_missing'] = (
        nfip[['is_lag_missing', 'is_elev_diff_missing', 'is_bfe_missing']].max(axis=1))

print('Missingness indicator agreement:')
print(nfip[['is_lag_missing', 'is_elev_diff_missing',
            'is_bfe_missing', 'is_elev_missing']].mean().round(4))
print('\nPairwise agreement rate:')
print('  elev_diff vs bfe:',
      (nfip['is_elev_diff_missing'] == nfip['is_bfe_missing']).mean().round(4))
print('  elev_diff vs lag:',
      (nfip['is_elev_diff_missing'] == nfip['is_lag_missing']).mean().round(4))

# Fill AFTER flagging. Genuine measured zeros exist and are distinguished from
# filled zeros by the indicator.
for c in ELEV_COLS:
    n_real_zero = ((nfip[c] == 0) & nfip[c].notna()).sum()
    n_filled = nfip[c].isna().sum()
    nfip[c] = nfip[c].fillna(0.0)
    print(f'{c}: {n_real_zero:,} genuine zeros retained, {n_filled:,} filled to 0.')

Missingness indicator agreement:
is_lag_missing          0.6568
is_elev_diff_missing    0.6447
is_bfe_missing          0.6518
is_elev_missing         0.6649
dtype: float64

Pairwise agreement rate:
  elev_diff vs bfe: 0.9918
  elev_diff vs lag: 0.9768
elevationDifference: 19,067 genuine zeros retained, 188,314 filled to 0.
baseFloodElevation: 1,021 genuine zeros retained, 190,384 filled to 0.
lowestAdjacentGrade: 2,563 genuine zeros retained, 191,870 filled to 0.
lowestFloorElevation: 363 genuine zeros retained, 191,122 filled to 0.


## 7. NHGIS median home value — compile

In [ ]:
def build_med_price_df(nhgis_dir):
    frames = []
    for path in glob.glob(os.path.join(nhgis_dir, '*_blck_grp.csv')):
        m = re.search(r'_(\d{4})5_blck_grp\.csv', os.path.basename(path))
        if not m:
            continue
        uw_year = int(m.group(1)) + 2   # 2005-2009 ACS -> uw_year 2011

        df = pd.read_csv(path, encoding='latin1', low_memory=False)
        desc = df.iloc[0]
        val_col = next((c for c in df.columns
                        if 'median' in str(desc[c]).lower()
                        and 'error' not in str(desc[c]).lower()), None)
        if val_col is None:
            print(f'  SKIPPED (no median column found): {os.path.basename(path)}')
            continue

        df = df.iloc[1:].reset_index(drop=True)
        tmp = df[['GISJOIN', val_col]].copy()
        tmp = tmp.rename(columns={val_col: 'median_home_value'})
        tmp['uw_year'] = uw_year

        # GISJOIN = G + state(2) + '0' + county(3) + '0' + tract(6) + blockgroup(1)
        f = tmp['GISJOIN'].str.replace('G', '', regex=False)
        tmp['censusBlockGroupFips'] = f.str[0:2] + f.str[3:6] + f.str[7:]

        frames.append(tmp[['uw_year', 'censusBlockGroupFips', 'median_home_value']])

    out = pd.concat(frames, ignore_index=True)
    out['median_home_value'] = pd.to_numeric(out['median_home_value'], errors='coerce')
    return out

med_price = build_med_price_df(NHGIS_DIR)
print(f'Compiled {len(med_price):,} block-group / year records.')
print(f'Years present: {sorted(med_price["uw_year"].unique())}')

# Census codes. The jam value is missing data; the top-code is a real censored
# observation and must NOT be dropped -- doing so would systematically delete
# high-value coastal tracts from a flood severity model.
n_jam = (med_price['median_home_value'] == ACS_JAM_VALUE).sum()
n_top = (med_price['median_home_value'] == ACS_TOP_CODE).sum()
print(f'\nACS jam values ({ACS_JAM_VALUE:.0f}): {n_jam:,} -> recoded to NaN')
print(f'ACS top-codes ({ACS_TOP_CODE:.0f}): {n_top:,} -> retained + flagged')

med_price.loc[med_price['median_home_value'] == ACS_JAM_VALUE,
              'median_home_value'] = np.nan
med_price['is_home_value_topcoded'] = (
    med_price['median_home_value'] == ACS_TOP_CODE).astype(int)

assert med_price['median_home_value'].min() >= 0 or med_price['median_home_value'].isna().all(), \
    'Negative median_home_value survived jam-value recoding'
print(f'\nPost-recode range: {med_price["median_home_value"].min():,.0f} to '
      f'{med_price["median_home_value"].max():,.0f}')

Compiled 3,626,204 block-group / year records.
Years present: [np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]

ACS jam values (-666666666): 48,184 -> recoded to NaN
ACS top-codes (1000001): 14,159 -> retained + flagged

Post-recode range: 5,400 to 2,000,001


## 8. Temporal imputation and merge

In [ ]:
all_fips  = med_price['censusBlockGroupFips'].unique()
all_years = range(med_price['uw_year'].min(), med_price['uw_year'].max() + 1)
full_index = pd.MultiIndex.from_product([all_fips, all_years],
                                        names=['censusBlockGroupFips', 'uw_year'])

grid = (med_price.set_index(['censusBlockGroupFips', 'uw_year'])
        .reindex(full_index).reset_index()
        .sort_values(['censusBlockGroupFips', 'uw_year']))

g = grid.groupby('censusBlockGroupFips')['median_home_value']
grid['median_home_value'] = g.ffill()
grid['median_home_value'] = grid.groupby('censusBlockGroupFips')['median_home_value'].bfill(limit=1)
grid['is_home_value_topcoded'] = grid.groupby('censusBlockGroupFips')['is_home_value_topcoded'].ffill().fillna(0)

max_med_year = grid['uw_year'].max()
nfip['merge_uw_year'] = nfip['uw_year'].clip(upper=max_med_year)

merged = nfip.merge(
    grid, how='left',
    left_on=['censusBlockGroupFips', 'merge_uw_year'],
    right_on=['censusBlockGroupFips', 'uw_year'],
    suffixes=('', '_grid'),
)
merged = (merged.drop(columns=['merge_uw_year', 'uw_year_grid'])
          if 'uw_year_grid' in merged.columns else merged.drop(columns=['merge_uw_year']))

print(f'Merged: {len(merged):,} records.')
print(f'median_home_value missing after temporal merge: '
      f'{merged["median_home_value"].isna().sum():,} '
      f'({merged["median_home_value"].isna().mean() * 100:.2f}%)')

Merged: 292,109 records.
median_home_value missing after temporal merge: 38,235 (13.09%)


## 9. Spatial imputation

In [ ]:
grid['tract_fips']  = grid['censusBlockGroupFips'].str[:11]
grid['county_fips'] = grid['censusBlockGroupFips'].str[:5]

tract_med  = grid.groupby(['tract_fips', 'uw_year'])['median_home_value'].median().reset_index()
county_med = grid.groupby(['county_fips', 'uw_year'])['median_home_value'].median().reset_index()

merged['tract_fips']    = merged['censusBlockGroupFips'].astype(str).str[:11]
merged['county_fips']   = merged['censusBlockGroupFips'].astype(str).str[:5]
merged['merge_uw_year'] = merged['uw_year'].clip(upper=max_med_year)
merged['imputation']    = pd.Series(['none'] * len(merged), index=merged.index, dtype=object)

for level, table, key in [('tract', tract_med, 'tract_fips'),
                          ('county', county_med, 'county_fips')]:
    merged = merged.merge(
        table, how='left',
        left_on=[key, 'merge_uw_year'], right_on=[key, 'uw_year'],
        suffixes=('', f'_{level}'),
    )
    fill_col = f'median_home_value_{level}'
    mask = merged['median_home_value'].isna() & merged[fill_col].notna()
    merged.loc[mask, 'median_home_value'] = merged.loc[mask, fill_col]
    merged.loc[mask, 'imputation'] = level
    merged = merged.drop(columns=[fill_col, f'uw_year_{level}'])

merged = merged.drop(columns=['tract_fips', 'county_fips', 'merge_uw_year'])

print('Imputation source:')
print(merged['imputation'].value_counts())

before = len(merged)
merged = merged.dropna(subset=['median_home_value'])
print(f'\nDropped {before - len(merged):,} records with unimputable home value.')
print(f'Remaining: {len(merged):,}')

Imputation source:
imputation
none      254894
county     32489
tract       4726
Name: count, dtype: int64

Dropped 1,020 records with unimputable home value.
Remaining: 291,089


## 10. Merge county classification

In [ ]:
c_code = merged['countyCode'].astype(str).str.replace(r'\.0$', '', regex=True)
c_code = c_code.where(c_code != 'nan').str.zfill(5)
bg_code = merged['censusBlockGroupFips'].astype(str).str[:5]
bg_code = bg_code.where(bg_code != 'nan')
merged['merge_fips'] = c_code.fillna(bg_code)

merged = merged.merge(county_classes_df, how='left',
                      left_on='merge_fips', right_on='FIPS')

unmatched = merged['COUNTY_NAME'].isna().sum()
print(f'Unmatched county FIPS: {unmatched:,} ({unmatched / len(merged) * 100:.2f}%)')
if unmatched / len(merged) > 0.05:
    warnings.warn('Over 5% of records failed the county merge -- check FIPS formats.',
                  UserWarning)

merged = merged.drop(columns=['merge_fips']).dropna(subset=['COUNTY_NAME'])
print(f'Records after dropping unmatched counties: {len(merged):,}')

if not SPATIAL_FLAGS_TRUSTED:
    print('\nWARNING: SHORELINE_FLAG / WATERSHED_FLAG failed validation in section 2. '
          'They are written to the parquet but must not be used in modelling until '
          'county_classification.csv is rebuilt.')

Unmatched county FIPS: 216 (0.07%)
Records after dropping unmatched counties: 290,873



## 11. Validation suite

Every check that would have caught one of the defects found during modelling.
Failures raise here rather than surfacing as a convergence error later.

In [ ]:
problems = []

def check(name, condition, detail=''):
    status = 'PASS' if condition else 'FAIL'
    if not condition:
        problems.append(f'{name}: {detail}')
    print(f'  [{status}] {name}' + (f'  -- {detail}' if detail and not condition else ''))

print('--- Validation ---')

for c in ELEV_COLS:
    mx = merged[c].abs().max()
    check(f'{c} within plausible range', mx < ELEV_SENTINEL_MIN,
          f'max |value| = {mx:,.1f}, sentinel threshold {ELEV_SENTINEL_MIN:,.0f}')

check('median_home_value non-negative',
      merged['median_home_value'].min() >= 0,
      f'min = {merged["median_home_value"].min():,.0f}')

check('median_home_value no jam values',
      (merged['median_home_value'] == ACS_JAM_VALUE).sum() == 0)

check('no null in modelling columns',
      merged[['buildingDamageAmount', 'floodZoneCurrent',
              'basementEnclosureCrawlspaceType', 'median_home_value']].isna().sum().sum() == 0)

for c in ('floodZoneCurrent', 'basementEnclosureCrawlspaceType', 'obstructionType'):
    counts = merged[c].value_counts()
    check(f'{c} has no cell under 1000', counts.min() >= 1000,
          f'smallest level "{counts.idxmin()}" has {counts.min():,} records')

check('buildingAge plausible',
      merged['buildingAge'].max() < 250 if merged['buildingAge'].notna().any() else True,
      f'max = {merged["buildingAge"].max():.1f} years')

check('positive-damage subset is usable for Gamma',
      (merged['buildingDamageAmount'] > 0).sum() > 100_000,
      f'{(merged["buildingDamageAmount"] > 0).sum():,} positive records')

check('SHORELINE_FLAG validated', SPATIAL_FLAGS_TRUSTED,
      'county_classification.csv failed coastal spot-checks -- rebuild required')

print()
if problems:
    print(f'{len(problems)} validation problem(s):')
    for p in problems:
        print(f'  - {p}')
else:
    print('All validation checks passed.')

--- Validation ---
  [PASS] elevationDifference within plausible range
  [PASS] baseFloodElevation within plausible range
  [PASS] lowestAdjacentGrade within plausible range
  [PASS] lowestFloorElevation within plausible range
  [PASS] median_home_value non-negative
  [PASS] median_home_value no jam values
  [PASS] no null in modelling columns
  [PASS] floodZoneCurrent has no cell under 1000
  [PASS] basementEnclosureCrawlspaceType has no cell under 1000
  [PASS] obstructionType has no cell under 1000
  [PASS] buildingAge plausible
  [PASS] positive-damage subset is usable for Gamma
  [FAIL] SHORELINE_FLAG validated  -- county_classification.csv failed coastal spot-checks -- rebuild required

1 validation problem(s):
  - SHORELINE_FLAG validated: county_classification.csv failed coastal spot-checks -- rebuild required


## 12. Export

In [ ]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

merged['imputation'] = merged['imputation'].fillna('none').astype(str)
for c in ('median_home_value', 'buildingDamageAmount', 'buildingReplacementCost',
          'buildingPropertyValue'):
    merged[c] = pd.to_numeric(merged[c], errors='coerce')

merged.to_parquet(OUT_PATH, index=False)
print(f'Wrote {len(merged):,} records x {merged.shape[1]} columns to:')
print(f'  {OUT_PATH}')

print('\nColumns:')
print(sorted(merged.columns.tolist()))

print('\nModelling notes for the downstream notebook:')
print('  - Filter buildingDamageAmount > 0 for Gamma / Inverse Gaussian / lognormal.')
print('  - Use is_elev_missing (consolidated); the three component flags are 99.6%')
print('    identical and will destabilise the fit if entered together.')
print('  - Scale median_home_value by 1e5 so the coefficient reads per $100k and the')
print('    design matrix stays well-conditioned.')
print('  - Fit with method="newton"; the IRLS default exhibits a limit cycle on this')
print('    design and never satisfies the deviance tolerance.')
print('  - Exclude SHORELINE_FLAG / WATERSHED_FLAG until validation passes.')

Wrote 290,873 records x 43 columns to:
  /content/drive/MyDrive/Programming/nfip/Single-Residence Severity/data/processed/model_ready_nfip.parquet

Columns:
['COUNTY_NAME', 'FIPS', 'SHORELINE_FLAG', 'STATE', 'WATERSHED_FLAG', 'amountPaidOnBuildingClaim', 'asOfDate', 'baseFloodElevation', 'basementEnclosureCrawlspaceType', 'buildingAge', 'buildingDamageAmount', 'buildingPropertyValue', 'buildingReplacementCost', 'censusBlockGroupFips', 'censusTract', 'countyCode', 'dateOfLoss', 'elevatedBuildingIndicator', 'elevationDifference', 'floodZoneCurrent', 'floodproofedIndicator', 'imputation', 'is_bfe_missing', 'is_elev_diff_missing', 'is_elev_missing', 'is_home_value_topcoded', 'is_lag_missing', 'is_lfe_missing', 'is_mobile_home', 'is_zero_damage', 'lowestAdjacentGrade', 'lowestFloorElevation', 'median_home_value', 'netBuildingPaymentAmount', 'numberOfFloorsInTheInsuredBuilding', 'obstructionType', 'originalConstructionDate', 'originalNBDate', 'postFIRMConstructionIndicator', 'primaryResidenc